# **Deeploc 2.1**

2025 겨울 URP / 문서연, 김대현, 권효재


---


여기서부턴 Deeploc 2.1과 같은 구조로 학습을 진행!


---


개선/공부 필요(2026_01_18):

---

# **라이브러리 설명**

# torch
Pytorch 라이프러리 패키지


* torch.autograd: 자동 미분을 위한 함수가 포함됨(ex. enable/no_grad: 자동 미분 on/off, Function: 자체 미분 함수 정의 클래스)
* torch.nn: 신경망 구축을 위한 기본 데이터 구조/레이어(RNN/LSTM)/활성화 함수(ReLU)/손실 함수(MSELoss) 포함됨
* torch.optim: 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 중심의 파라미터 옵티마이저 알고리즘
* torch.utils.data: SDG 반복연산 시에 사용하는 미니배치 유틸리티 포함됨
* torch.onnx: ONNX(Open Neural Network Exchange) 포맷으로 모델 export 할 때 사용

In [1]:
# 필요 라이브러리 설치 및 import
!pip install -q torch pandas numpy safetensors optuna scikit-learn #lmdb

import io
import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import math

from tqdm import tqdm
from safetensors import safe_open
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, classification_report, precision_score, recall_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 21.0 MB/s eta 0:00:00


In [2]:
# <저장소 설정>

# Colab 환경
from google.colab import drive

drive.mount("/content/drive")
SAVE_PATH = "/content/drive/MyDrive/Github/Mprotein_hydrophobic/dataset/etc/embeddings_safetensors"

# 로컬 환경
# SAVE_PATH = "./"

os.makedirs(SAVE_PATH, exist_ok=True)

Mounted at /content/drive


In [ ]:
# 데이터셋 클래스 정의
class K_CV_Dataset(Dataset):
    # 데이터셋 전처리
    def __init__(self, K_CV, validation_k, save_path, is_train=True):
        # 기본 저장 경로 & 핸들 초기화
        self.save_path = save_path
        self.path_x = os.path.join(self.save_path, f"embeddings.safetensors")
        self.path_y = os.path.join(self.save_path, f"targets.safetensors")
        self.handle_x = None
        self.handle_y = None

        # 훈련/테스트 > 훈련 파티션 숫자 리스트 만들기
        if is_train:
            self.train_part = [i for i in range(K_CV) if i != validation_k]
        else:
            self.train_part = [validation_k]

        # 키 필터링 + 재현성을 위해 정렬
        self.train_keys = []
        # self.targets_dict = {}
        with safe_open(self.path_y, framework="pt") as f:
            all_keys = f.keys()
            for p in self.train_part:
                prefix = f"part_{p}_"
                p_keys = [k for k in all_keys if k.startswith(prefix)]
                self.train_keys.extend(p_keys)
                # for k in p_keys:
                #     self.targets_dict[k] = f.get_tensor(k)

        self.train_keys.sort()

    # 몇개있는지 알려줘야함
    def __len__(self):
        return len(self.train_keys)

    def open_files(self):
        # 멀티프로세싱의 각 worker 안에서 파일을 처음 한 번만 엽니다. > 뭔말인지 잘 이해 못함 솔직히 num worker 각각이 핸들 한번씩 연다는말같긴한데..
        if self.handle_x is None:
            self.handle_x = safe_open(self.path_x, framework="pt")
        if self.handle_y is None:
            self.handle_y = safe_open(self.path_y, framework="pt")

    def __del__(self):
        self.handle_x = None
        self.handle_y = None

        if self.handle_x is not None:
            self.handle_x = None
        if self.handle_y is not None:
            self.handle_y = None

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        self.open_files()

        # 리스트에서 idx번째 key 호출
        key = self.train_keys[idx]

        # get_slice().asarray() 또는 get_tensor() 사용
        # Safetensors는 메모리 매핑을 쓰기 때문에 이 과정이 매우 빠릅니다.
        embedding = self.handle_x.get_tensor(key)
        # target = self.targets_dict[key]
        target = self.handel_y.get_tensor(key)

        return embedding, target


# 작동 테스트
# c = K_CV_Dataset(4, 0, SAVE_PATH)
# a, b = c[0]
# print(a)
# print(b)

In [13]:
import time

c = K_CV_MultipleFiles_Dataset(4, 0, SAVE_PATH)

# 테스트1: 순차 접근
st = time.time()
for i in range(100):
    c[i]
print(f"순차 100개: {time.time()-st:.2f}s")

# 테스트2: 셔플 접근
import random

random.shuffle(indices := list(range(len(c))))
st = time.time()
for i in indices[:100]:
    c[i]
print(f"셔플 100개: {time.time()-st:.2f}s")

순차 100개: 2.12s
셔플 100개: 112.87s


In [4]:
def padding_collate_fn(batch):
    # 임베딩/타겟 분리
    embeddings = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    # 2. 임베딩 패딩
    padded_embeddings = pad_sequence(embeddings, batch_first=True, padding_value=0)
    targets = torch.stack(targets)
    return padded_embeddings, targets

In [5]:
# 앞에서 Layer normalization 해준걸 인풋으로 받는다고 가정
class MultiheadAttentionPooling(nn.Module):
    def __init__(self, attn_dim=128, heads_nums=2, kernel_size=5):
        super().__init__()
        self.attn_dim = attn_dim
        # 우선은,, 멀티헤드로 구현을 한다
        self.heads_nums = heads_nums
        # 멀티헤드의 디멘션은 전체 디멘션을 헤드 개수로 나눈것
        # 이거때문에 헤드 개수를 잘 나눠지도록(?) 설정함 보통
        self.head_dim = attn_dim // heads_nums

        # Q, K = V 만들기
        # Q : learnable Query, 먼저 (1, 1, attn_dim) 에 해당하는 빈 벡터 > xavier_uniform_ 하면 입출력 고려해서 난수생성 가능
        self.query = nn.Parameter(torch.empty(1, 1, attn_dim))
        nn.init.xavier_uniform_(self.query.data)
        self.w_kv = nn.Linear(
            attn_dim, attn_dim
        )  # 입력/출력 크기가 attn_dim인 Linear FC를 수행하는 모듈 // key value 짜피 같으니까 이걸로 한번에 할거고 논문도 그렇게 했는데 둘이 따로 초기화한다면? 즉 파라미터가 두개라면..?

    def forward(self, x):
        # 입력으로 받을 형태: (batch_size, sequence_length, attn_dim)
        batch_size, sequence_length, _ = x.shape

        # 배치 사이즈에 맞게 복제: 각 배치 샘플 전부 같은 쿼리 파라미터 공유
        # attn_dim을 헤드별로 쪼갬()
        # 헤드별로 계산할거라서 헤드를 앞으로 뺌
        Q = (
            self.query.repeat(batch_size, 1, 1)
            .view(batch_size, 1, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        K = (
            self.w_kv(x)
            .view(batch_size, sequence_length, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        V = (
            self.w_kv(x)
            .view(batch_size, sequence_length, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        # 셋다 (batch_size, head_nums, sequence_length(Q:1; per token), head_dim)
        # print(Q)
        # print(K)
        # print(V)
        # 행렬곱 > Attention Scalar score 구함!
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # print(f"어텐션 Score: {scores.shape}")
        # Conv1d 가우시안 필터
        scores_c1d_gaussian = scores

        # Softmax 적용
        attn_weights = F.softmax(scores_c1d_gaussian, dim=-1)
        # print(f"어텐션 Weights: {attn_weights.shape}")

        # 행렬곱 > weighted attention
        attnpooled_nosum = torch.matmul(attn_weights, V)
        # print(f"어텐션 Pooling (합하기 전): {attnpooled_nosum.shape}")

        # 가중합을 위해 (batch_size, head_nums, 1, head_dim) 순서니까
        # 1 없애고 reshape로 head concat해주기
        attentionpooled = attnpooled_nosum.squeeze(2).reshape(batch_size, self.attn_dim)
        # print(f"어텐션 Pooling: {attentionpooled.shape}")

        return attentionpooled

In [ ]:
# import matplotlib.pyplot as plt

# sigma1 = 3
# sigma2 = 50


# def gaussian_filter1d(size, sigma):
#     filter_range = np.linspace(-int(size / 2), int(size / 2), size)
#     gaussian_filter = [
#         1 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-(x**2) / (2 * sigma**2))
#         for x in filter_range
#     ]
#     return gaussian_filter


# fig, ax = plt.subplots(1, 2)
# ax[0].plot(gaussian_filter1d(size=365, sigma=sigma1))
# ax[0].set_title(f"sigma= {sigma1}")
# ax[1].plot(gaussian_filter1d(size=365, sigma=sigma2))
# ax[1].set_title(f"sigma= {sigma2}")
# plt.show()

In [6]:
class Deeploc2_1(nn.Module):
    def __init__(self, heads_nums, embedding_dim, attn_dim, output_dim=4, hidden_dim=0):
        super().__init__()
        self.input_layer_normalization = nn.LayerNorm(embedding_dim)
        self.input_linear_fc = nn.Linear(embedding_dim, attn_dim)
        self.attention_layer_normalization = nn.LayerNorm(attn_dim)
        self.attention_head = MultiheadAttentionPooling(
            attn_dim=attn_dim, heads_nums=heads_nums
        )
        self.dropout = nn.Dropout(0.1)
        if hidden_dim == 0:
            hidden_dim = attn_dim
        self.output_linear_fc = nn.Linear(
            hidden_dim, output_dim
        )  # mlp층 쌓으면 attn_dim 아니고 hidden_dim 됨 이거 나중에 고쳐야할듯

    def forward(self, x):
        # layer normalization, 선형층 통과 > embedding dim에서 attn dim으로,,
        x = self.input_layer_normalization(x)
        x = self.input_linear_fc(x)
        # attention head 통과 > (batch_size, attn_dim)
        x = self.attention_layer_normalization(x)
        x = self.attention_head(x)
        x = self.dropout(x)
        # 선형층 통과(분류기) > attn dim에서 4개로,,
        x = self.output_linear_fc(x)
        return x

In [15]:
def multiclass_focal_loss_onehot(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    alpha: torch.Tensor = None,
    gamma: float = 2.0,
    reduction: str = "mean",
) -> torch.Tensor:

    # Softmax to get probabilities
    p = F.softmax(inputs, dim=1)  # (N, C)

    # Cross entropy: -sum(targets * log(p))
    ce_loss = -(targets * torch.log(p + 1e-7)).sum(dim=1)  # (N,)

    # p_t: probability of true class
    p_t = (targets * p).sum(dim=1)  # (N,)

    # Focal weight
    focal_weight = (1 - p_t) ** gamma  # (N,)

    # Focal loss
    focal_loss = focal_weight * ce_loss  # (N,)

    # Apply alpha
    if alpha is not None:
        if isinstance(alpha, (list, tuple)):
            alpha = torch.tensor(alpha, device=inputs.device)
        alpha = alpha.to(inputs.device)

        # alpha_t: alpha for true class
        alpha_t = (targets * alpha).sum(dim=1)  # (N,)
        focal_loss = alpha_t * focal_loss

    # Reduction
    if reduction == "mean":
        return focal_loss.mean()
    elif reduction == "sum":
        return focal_loss.sum()
    return focal_loss

In [16]:
def get_effective_number(samples_per_class, beta=0.9999):
    effective_num = 1.0 - torch.pow(beta, samples_per_class)
    weights = (1.0 - beta) / effective_num
    weights = weights / weights.sum() * len(samples_per_class)
    return weights

In [17]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K_CV = 4
NUM_EPOCHS = 12
total_class_counts = torch.tensor([
        2110.0,   # peripheral
        17069.0,  # soluble
        6982.0,   # TM
        788.0     # LipidAnchor
    ])
alpha = get_effective_number(total_class_counts, beta=0.99)

In [19]:
DATASET_TRAIN = K_CV_MultipleFiles_Dataset(K_CV, 3, SAVE_PATH)
DATASET_TEST = K_CV_MultipleFiles_Dataset(K_CV, 3, SAVE_PATH, is_train=False)
DATALOADER_TRAIN = DataLoader(
    DATASET_TRAIN,
    batch_size=5,
    shuffle=False,
    num_workers=2,
    collate_fn=padding_collate_fn,
    pin_memory=True,
    persistent_workers=True,
)
DATALOADER_TEST = DataLoader(
    DATASET_TEST,
    batch_size=5,
    shuffle=False,
    num_workers=2,
    collate_fn=padding_collate_fn,
    pin_memory=True,
    persistent_workers=True,
)
MODEL = Deeploc2_1(embedding_dim=1152, attn_dim=128, output_dim=4, heads_nums=2).to(
    DEVICE
)
OPTIMIZER = AdamW(MODEL.parameters(), lr=1e-4)

total_class_counts = torch.tensor([
        2110.0,   # peripheral
        17069.0,  # soluble
        6982.0,   # TM
        788.0     # LipidAnchor
    ])

for epoch in range(NUM_EPOCHS):
    # TRAIN
    MODEL.train()
    for embeddings, targets in tqdm(DATALOADER_TRAIN):
        OPTIMIZER.zero_grad()
        embeddings, targets = embeddings.to(DEVICE), targets.to(DEVICE)
        outputs = MODEL(embeddings)
        loss_value = multiclass_focal_loss_onehot(
            outputs, targets, alpha=alpha, gamma=1.5, reduction="mean"
        )
        loss_value.backward()
        OPTIMIZER.step()

    # TEST

    # ========== 검증(Validation) 단계 ==========
    MODEL.eval()  # 모델을 평가 모드로 전환 (dropout 등 비활성화)

    # 검증에 필요한 변수들 초기화
    val_loss = 0.0  # 전체 손실의 합 (나중에 평균 계산용)
    all_probs = []  # 모든 배치의 예측 확률값 저장
    all_targets = []  # 모든 배치의 정답 라벨 저장

    # 그래디언트 계산 비활성화 (메모리 절약 및 속도 향상)
    with torch.no_grad():
        # 테스트 데이터로더에서 배치 단위로 데이터 가져오기
        for embeddings, targets in DATALOADER_TEST:
            # 데이터를 GPU로 이동
            embeddings = embeddings.to(DEVICE)
            targets = targets.to(DEVICE)

            # 모델에 입력하여 예측값 얻기 (logits 형태)
            outputs = MODEL(embeddings)

            # 손실 계산 (reduction="mean"으로 배치 내 평균 계산)
            batch_loss = multiclass_focal_loss_onehot(
                outputs, targets, alpha=alpha, gamma=1.5, reduction="mean"
            )
            val_loss += batch_loss.item()  # 배치 손실을 누적

            # 모델 출력값을 확률로 변환 (sigmoid 적용)
            # outputs는 logits이므로 sigmoid를 적용하면 0~1 사이의 확률이 됨
            probs = torch.sigmoid(outputs)

            # CPU로 이동 후 numpy 배열로 변환하여 리스트에 저장
            all_probs.append(probs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # ========== 전체 데이터를 하나로 합치기 ==========
    # 배치별로 나뉘어 있던 예측값과 정답을 하나의 배열로 합침
    all_probs = np.concatenate(all_probs, axis=0)    # shape: (전체샘플수, 4)
    all_targets = np.concatenate(all_targets, axis=0)  # shape: (전체샘플수, 4)

    # ========== 예측값을 0 또는 1로 변환 ==========
    # 확률이 0.5 이상이면 1, 미만이면 0으로 예측
    all_preds = (all_probs >= 0.5).astype(int)  # shape: (전체샘플수, 4)

    # ========== 평가 지표 계산 ==========
    # 1. Accuracy: 모든 라벨이 정확히 일치하는 샘플의 비율 (매우 엄격한 지표)
    #    예: 정답이 [1,0,1,0]이고 예측이 [1,0,1,0]이면 1, 하나라도 다르면 0
    acc = accuracy_score(all_targets, all_preds)

    # 2. F1-Score (Macro): 각 클래스별 F1 점수를 구한 후 평균
    #    - 클래스별로 동등하게 중요하게 취급
    #    - 클래스 불균형이 있을 때 유용
    f1_macro = f1_score(all_targets, all_preds, average="macro")

    # 3. F1-Score (Micro): 전체 샘플의 TP, FN, FP를 합쳐서 계산
    #    - 전체적인 성능을 보는 지표
    f1_micro = f1_score(all_targets, all_preds, average="micro")

    # 중간 결과 출력
    print(
        f"Accuracy: {acc:.4f} | F1 (Macro): {f1_macro:.4f} | F1 (Micro): {f1_micro:.4f}"
    )

    # 4. MCC (Matthews Correlation Coefficient): 각 클래스별로 계산 후 평균
    #    - -1 ~ 1 사이의 값, 1에 가까울수록 좋음
    #    - 클래스 불균형에 강건한 지표
    mcc_scores = []
    num_classes = all_targets.shape[1]  # 클래스 개수 (4개)
    for class_idx in range(num_classes):
        # 각 클래스별로 MCC 계산
        # all_probs[:, class_idx]: 해당 클래스의 모든 샘플에 대한 확률
        # all_targets[:, class_idx]: 해당 클래스의 모든 샘플에 대한 정답
        class_preds = (all_probs[:, class_idx] >= 0.5).astype(int)
        class_targets = all_targets[:, class_idx]
        mcc = matthews_corrcoef(class_targets, class_preds)
        mcc_scores.append(mcc)
    avg_mcc = np.mean(mcc_scores)  # 클래스별 MCC의 평균

    # ========== 검증 손실의 평균 계산 ==========
    # 배치별 평균 손실을 더한 후 배치 개수로 나누어 전체 평균 계산
    val_loss_mean = val_loss / len(DATALOADER_TEST)

    # ========== 최종 결과 출력 ==========
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Val Loss: {val_loss_mean:.4f} | "
        f"Avg MCC: {avg_mcc:.4f} | "
        f"F1-Macro: {f1_macro:.4f} | "
        f"Acc: {acc:.4f}"
    )
del MODEL, DATALOADER_TRAIN, DATALOADER_TEST
torch.cuda.empty_cache()

100%|██████████| 3273/3273 [04:10<00:00, 13.09it/s]


KeyboardInterrupt: 

In [3]:
# 데이터셋 클래스 정의
class K_CV_MultipleFiles_Dataset(Dataset):
    # 데이터셋 전처리
    def __init__(self, K_CV, validation_k, save_path, is_train=True):
        # 훈련 > K개의 데이터셋 중 validation_k 제외한 리스트 생성
        # 테스트 > validation_k만 생성
        if is_train:
            self.train_part = [i for i in range(K_CV) if i != validation_k]
        else:
            self.train_part = [validation_k]

        # 기본 저장 경로: save_path
        self.save_path = save_path

        # x와 y는 같은 key(ACC) 공유
        # y로부터 총합 key list를 만들 예정
        # key가 x의 어떤 파티션에 있는지 불러오기 위해 key와 x 파일 주소가 저장된 일종의 주소록 딕셔너리 생성
        # 반복문을 돌며 총합 targets 딕셔너리 생성
        self.key_list = []
        self.targets = {}
        self.keys_to_x = {}
        self.file_handels = {}

        for i in self.train_part:
            # x, y 경로 설정
            path_x = os.path.join(self.save_path, f"embeddings_part_{i}.safetensors")
            path_y = os.path.join(self.save_path, f"target_part_{i}.pt")

            ##타겟 파일
            # 타겟 파일 로드, 딕셔너리 업데이트, keys 리스트 가져오기
            y = torch.load(path_y, weights_only=False)
            self.targets.update(y)
            key_list_k = list(y.keys())
            self.key_list.extend(key_list_k)

            ##딕셔너리 주소 저장
            for key in key_list_k:
                self.keys_to_x[key] = path_x

        # 재현성을 위해 정렬
        self.key_list.sort()

    # 몇개있는지 알려줘야함
    def __len__(self):
        return len(self.key_list)

    def open_files(self, path):
        if path not in self.file_handels:
            self.file_handels[path] = safe_open(
                path, framework="pt", device="cpu"
            )
        # 이 함수가 핸들을 반환하도록 변경
        return self.file_handels[path]

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        # 리스트에서 idx번째 key 호출
        key = self.key_list[idx]
        path_x_k = self.keys_to_x[key]

        # 2. 파일 핸들이 열려있는지 확인하고, 없으면 엽니다.
        f = self.open_files(path_x_k)

        embedding = f.get_tensor(key).squeeze().clone()

        target_numpy = self.targets[key]
        target = torch.from_numpy(target_numpy).squeeze(0).float()

        return embedding, target


# 작동 테스트
c = K_CV_MultipleFiles_Dataset(4, 0, SAVE_PATH)
a, b = c[0]
print(a)
print(b)

tensor([[  74.2471,   51.3684,  -66.2884,  ...,    3.2179,   15.8106,
          -32.0539],
        [  38.9175,   37.7472,  -60.0651,  ...,    5.7212,   59.3637,
          -83.0072],
        [  -8.8964,   65.5401,  -28.7627,  ...,  -28.3236,  122.1794,
          -33.1644],
        ...,
        [ -62.1385,  -25.0254,  -59.2227,  ...,   48.2738,  204.1275,
           -4.6377],
        [-150.7953,  -24.0576,  -39.7490,  ...,   -8.5499,  -82.4795,
          -33.1312],
        [ -62.2803,   40.8674,   70.2387,  ...,  -54.6412,   46.6652,
           13.9173]])
tensor([0., 0., 0., 1.])


In [4]:
# # 데이터셋 클래스 정의
# # 훈련 > K개의 데이터셋 중 validation_k 제외한 리스트 생성
# # 테스트 > validation_k만 생성
# class KCVDataset(Dataset):
#     # 초기화: 데이터셋 전처리
#     def __init__(self, K_CV, validation_k, save_path, is_train=True):
#         if is_train:
#             self.train_part = [i for i in range(K_CV) if i != validation_k]
#         else:
#             self.train_part = [validation_k]

#         # 기본 저장 경로: save_path
#         self.save_path = save_path

#         # 반복문 > 파일 합 targets/embeddings 딕셔너리
#         self.targets = {}
#         self.embeddings = {}
#         for i in self.train_part:
#             # x, y 경로 설정
#             path_x = os.path.join(self.save_path, f"embeddings_part_{i}.safetensors")
#             path_y = os.path.join(self.save_path, f"target_part_{i}.pt")

#             # 타겟 파일 로드
#             y = torch.load(path_y, weights_only=False)
#             self.targets.update(y)

#             # 임베딩 파일
#             with safe_open(path_x, framework="pt", device="cpu") as f:
#                 x = {k: f.get_tensor(k).squeeze() for k in f.keys()}
#                 self.embeddings.update(x)

#         # 키 리스트 반환
#         self.key_list = list(self.targets)

#     # 몇개있는지 알려줘야함
#     def __len__(self):
#         return len(self.key_list)

#     # 데이터셋 샘플 1개 가져오기
#     def __getitem__(self, idx):
#         # 리스트에서 idx번째 key 호출
#         key = self.key_list[idx]

#         # 키에 해당하는 임베딩, 타겟 불러오기
#         embedding = self.embeddings[key].clone()
#         target = torch.from_numpy(self.targets[key]).squeeze(0).float()

#         return embedding, target


# # 작동 테스트
# c = KCVDataset(4, 0, SAVE_PATH)
# a, b = c[0]
# print(a)
# print(b)

ValueError: could not determine the shape of object type 'torch.storage.UntypedStorage'

In [ ]:
# # 데이터셋 클래스 정의
# class LMDBDataset_K_CV(Dataset):
#     # 데이터셋 전처리
#     def __init__(self, K_CV, test_k, save_path, is_train=True):
#         if is_train:
#             self.train_part = [i for i in range(K_CV) if i != test_k]
#         else:
#             self.train_part = [test_k]
#         # 기본 저장 경로: save_path
#         self.save_path = save_path
#         # lmdb에서 key list 호출
#         self.env = lmdb.open(
#             self.save_path, readonly=True, lock=False, readahead=False, meminit=False
#         )
#         with self.env.begin() as txn:
#             self.keys = pickle.loads(txn.get(b"__keys__"))

#     # 몇개있는지 알려줘야함
#     def __len__(self):
#         return len(self.keys)

#     # 데이터셋 샘플 1개 가져오기
#     def __getitem__(self, idx):
#         key = self.keys[idx]
#         with self.env.begin() as txn:
#             data = torch.load(io.BytesIO(txn.get(key.encode())))
#         return data["embedding"], data["target"]


# # c = LMDBDataset_K_CV(4, 0, SAVE_PATH)
# # a, b = c[0]
# # print(a)
# # print(b)